In [1]:
!which python

/root/miniconda3/bin/python


In [2]:
!python --version

Python 3.12.3


In [1]:
import sys
print(sys.executable)

/root/miniconda3/envs/llm_train/bin/python


In [2]:
import torch
import transformers
import datasets
print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)
print(torch.cuda.is_available())

/root/miniconda3/envs/llm_train/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.5.1+cu121
5.9.0
4.8.5
True


In [3]:
import os
import subprocess

result = subprocess.run(
    'bash -c "source /etc/network_turbo && env | grep -i proxy"',
    shell=True,
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"

print("proxy env:")
for k in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY", "HF_ENDPOINT", "HF_HUB_DISABLE_XET"]:
    print(k, os.environ.get(k))


proxy env:
http_proxy http://172.29.51.4:12798
https_proxy http://172.29.51.4:12798
HTTP_PROXY None
HTTPS_PROXY None
HF_ENDPOINT https://hf-mirror.com
HF_HUB_DISABLE_XET 1


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
import datasets

In [12]:
import os
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

In [5]:

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

dataset_path = "/root/autodl-tmp/llm_ads_project/data/ads_sft_demo"
output_dir = "/root/autodl-tmp/llm_ads_project/outputs/day3_sft"

os.makedirs(output_dir, exist_ok=True)

In [6]:
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# =========================
# 1. 加载 tokenizer 和模型
# =========================

print("\n===== 加载 tokenizer =====")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Qwen 一般有 eos_token，但有些模型没有 pad_token
# Trainer batch padding 时需要 pad_token


gpu: NVIDIA GeForce RTX 3090

===== 加载 tokenizer =====


In [7]:
print(tokenizer.pad_token, tokenizer.eos_token)

<|endoftext|> <|im_end|>


In [8]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_id
)
print("model device:", next(model.parameters()).device)
print("model dtype:", next(model.parameters()).dtype)
model.to("cuda")
print("model device:", next(model.parameters()).device)
print("model dtype:", next(model.parameters()).dtype)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2510.30it/s]


model device: cpu
model dtype: torch.bfloat16
model device: cuda:0
model dtype: torch.bfloat16


In [10]:
model.config.use_cache = False

In [13]:
print("\n===== 加载 Dataset =====")
dataset = load_from_disk(dataset_path)
print(dataset)
print("\n第一条 text：")
print(dataset[0]["text"])


===== 加载 Dataset =====
Dataset({
    features: ['instruction', 'input', 'output', 'messages', 'text'],
    num_rows: 10
})

第一条 text：
<|im_start|>system
你是一个广告算法助手，擅长根据广告商品信息分析目标受众、核心卖点和点击倾向。<|im_end|>
<|im_start|>user
根据广告商品信息，判断目标受众、核心卖点和点击倾向。

广告信息：商品：9.9元包邮无糖乌龙茶；卖点：0糖0脂、解腻、适合控糖；投放场景：小红书信息流。<|im_end|>
<|im_start|>assistant
目标受众：学生党、上班族、控糖人群；核心卖点：低价、健康、解腻；点击倾向：高。<|im_end|>



In [14]:
max_length = 512

def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    # Causal LM 训练中，labels 通常等于 input_ids
    # 模型学习预测下一个 token
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("\n===== Tokenize Dataset =====")
tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset.column_names,
)


===== Tokenize Dataset =====


Map: 100%|██████████| 10/10 [00:00<00:00, 604.17 examples/s]


In [15]:
dataset.column_names

['instruction', 'input', 'output', 'messages', 'text']

In [16]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

In [18]:
print(tokenized_dataset)
print("\n第一条 tokenized 样本 keys:", tokenized_dataset[0].keys())
print("input_ids length:", len(tokenized_dataset[0]["input_ids"]))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

第一条 tokenized 样本 keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 122


In [19]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [22]:
print(type(model))
print(model.loss_function)
print(model.loss_type)

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
<function ForCausalLMLoss at 0x7f696f0cb5b0>
ForCausalLM
